# 04 · Spatial Interpolation & Geostatistics

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ByMaxAnjos/LCZ4py/blob/main/notebooks/local/04_spatial_interpolation_geostats.en.ipynb)

This notebook is the deepest geostatistics stop in the `local/` series. Starting from hourly
air-temperature readings at 23 Berlin weather stations, we walk the full classical
geostatistical pipeline: quantify how temperature similarity decays with distance (the
**semivariogram**), use that structure to krige a continuous temperature surface, let the
**LCZ class act as external drift** so the interpolation respects urban form, and finally
**cross-validate** the result before trusting it for anything downstream.

> **Rendering note:** every figure in this notebook is drawn with LCZ4py's native static matplotlib renderer by default (300 dpi PNG/PDF/SVG/TIFF, no Plotly/Kaleido needed). Pass `renderer="plotly"` to a plotting function to get the interactive Plotly figure instead.

## Why geostatistics for urban temperature fields

A weather-station network is a sparse, irregular sample of a spatially continuous field.
Turning point observations into a wall-to-wall map requires a *model* of how the variable
varies in space — that model is what geostatistics provides.

**The semivariogram** is the central diagnostic. For every pair of stations, we compute half
the squared difference in their observed value (semivariance) and plot it against the
distance separating them. Under spatial autocorrelation — Tobler's first law, "everything is
related to everything else, but near things are more related than distant things" — nearby
station pairs should show low semivariance and distant pairs higher semivariance, up to a
point where it levels off. Fitting a theoretical curve (spherical, exponential, Gaussian) to
this empirical relationship yields three interpretable parameters:

- **Nugget** — semivariance at (near-)zero distance. Represents measurement noise and
  sub-grid-scale variability the network cannot resolve.
- **Sill** — the semivariance plateau at large distances, roughly the overall variance of the
  field once spatial correlation has fully decayed.
- **Range** — the distance at which the curve reaches the sill; beyond this, observations tell
  you almost nothing about each other.

**Ordinary Kriging** uses this fitted variogram to compute the *statistically optimal* (BLUP —
best linear unbiased predictor) weighted average of nearby stations at every grid cell, along
with a kriging variance that quantifies prediction uncertainty.

**Why plain kriging is not enough for cities.** Air temperature over an urban area is *not*
simply a smooth function of geographic distance — it is strongly structured by the Local
Climate Zone (LCZ) that a point sits in (Stewart & Oke 2012). A station in a Compact Midrise
LCZ (1) and one in a Low-rise Sparsely built LCZ (9) two kilometres apart can differ by several
degrees even though geographic distance alone would say they should look similar. **External
Drift Kriging (EDK)** solves this by fitting the drift (the deterministic part of the model) as
a function of LCZ class membership, then kriging only the *residuals* — the leftover spatial
structure that distance-based correlation can still explain after urban form is accounted for.
This is exactly the innovation of `lcz4r`'s interpolation functions (Anjos et al. 2025,
*Scientific Reports*, https://www.nature.com/articles/s41598-025-92000-0), and it typically
produces sharper, more physically realistic urban heat maps than distance-only kriging or
generic interpolators like IDW/RBF.

**Why cross-validate before trusting the map.** A kriged raster always *looks* smooth and
plausible — that is precisely the danger. Without leaving stations out and checking how well
the model predicts values it never saw, there is no way to know whether the interpolation is
capturing real spatial structure or just producing a confident-looking guess. Leave-one-out
cross-validation (LOOCV) — the approach `lcz_interp_eval` implements — is the only way to put a
number (RMSE, MAE) on "how much should I trust this map" before using it for heat-exposure
assessment, planning, or any other downstream decision.

Install `LCZ4py` with the geostatistics/plotting dependencies (`pykrige`, `matplotlib`, `polars`). `plotly` is only needed for the opt-in interactive renderer (`renderer="plotly"`).

In [1]:
# (already installed in this environment)
import LCZ4py  # noqa

## Setup: Berlin LCZ map + station data

We reuse the same Berlin setup as the rest of the `local/` series: `lcz_get_map(city="Berlin")`
for the classification raster, and `lcz_data_berlin.csv` (23 stations, hourly, 2019–2020) for
the observations. To keep every geostatistical step in this notebook fast and fully
reproducible on Colab's free tier, we subset to the four hottest afternoon hours of the 25 July
2019 European heatwave — plenty of spatial contrast to see LCZ structure in the temperature
field, with all 23 stations reporting.

In [ ]:
from IPython.display import display
from LCZ4py import lcz_get_map
import pandas as pd

lcz_map_path = lcz_get_map(city="Berlin")
print(lcz_map_path)

df = pd.read_csv("https://raw.githubusercontent.com/ByMaxAnjos/LCZ4py/main/lcz_data_berlin.csv")
df["date"] = pd.to_datetime(df["date"])

# 25 July 2019 heatwave afternoon: strong urban-rural contrast, all 23 stations reporting
df_event = df[(df["date"] >= "2019-07-25 12:00:00") & (df["date"] <= "2019-07-25 15:00:00")]
print(df_event.shape, df_event["station"].nunique(), "stations")
df_event.head()

## Step 1 — The empirical semivariogram: `lcz_variogram`

`lcz_variogram(x, data_frame, var, station_id, sp_res=100.0, model="Sph", n_lags=20,
max_dist=None)` computes pairwise semivariances between all stations reporting at the same
time step, bins them into `n_lags` distance classes, averages across time steps, and fits a
theoretical model (`"Sph"` spherical, `"Exp"` exponential, or `"Gau"` Gaussian). It returns a
`VariogramResult` with the empirical/fitted table (`.df`), the fitted `nugget`/`sill`/`range`
(`.params`), the model fit quality (`.r2`), and a ready-made matplotlib figure (`.plot`), static by default.

We run this *before* any interpolation — the fitted range tells us the distance beyond which
kriging weights should taper to near zero, and the nugget/sill ratio tells us how noisy the
network is relative to its true spatial signal.

In [ ]:
from LCZ4py import lcz_variogram

vgm = lcz_variogram(
    lcz_map_path, df_event, var="airT", station_id="station",
    model="Sph", n_lags=10, lang="en",
)
print(vgm.params)
print("R2:", round(vgm.r2, 3))
display(vgm.plot)

**Reading the plot:** each dot is one distance bin, sized by how many station
pairs fall in it; the dashed line is the fitted spherical model. A low **nugget** relative to
the **sill** means most of the variance is genuinely spatial (good news for kriging); a fitted
**range** on the order of a few kilometres is typical for an intra-city network and confirms
that Berlin's station spacing is fine enough to resolve LCZ-scale temperature contrasts. This
`model` (spherical, by default) and its parameters are exactly what feeds the kriging step
next — we do not need to pass `nugget`/`sill`/`range` by hand; `lcz_interp_map`/`lcz_krige`
refit the variogram internally per time step, but the diagnostics here confirm the fitted
`model` choice is reasonable for this dataset.

## Step 2 — Under the hood: `krige_predict`, `rbf_predict`, `idw_predict`

Before calling the high-level `lcz_interp_map` wrapper, it's worth seeing the three
scatter-to-grid primitives it dispatches to, directly, on a small hand-built grid. This is
where the geostatistics actually happens:

- **`krige_predict(df, x_col, y_col, z_col, lcz_col, grid_x, grid_y, model="Sph",
  enable_lcz_drift=True, nlags=20)`** — Ordinary Kriging via `pykrige`. When `lcz_col` is given
  and `enable_lcz_drift=True`, it first regresses the observations on one-hot LCZ-class
  dummies (the "external drift"), krige-interpolates the *residuals*, then adds the per-class
  drift back at each grid cell — this is External Drift Kriging. Returns a `KrigeResult` with
  `.prediction` and `.variance` grids.
- **`rbf_predict(x, y, z, grid_x, grid_y, kernel="thin_plate_spline")`** — a smooth
  Radial-Basis-Function scatter-interpolator (`scipy.interpolate.RBFInterpolator`). No
  variogram fitting, no LCZ drift — purely geometric smoothness.
- **`idw_predict(x, y, z, grid_x, grid_y, power=2.0)`** — Inverse Distance Weighting, the
  simplest deterministic baseline.

We project the four heatwave-hour stations to UTM 33N (Berlin's zone), sample their LCZ class
directly from the map raster, build a coarse 40×40 manual grid over the station bounding box,
and compare all three.

In [4]:
import numpy as np
import rasterio
from pyproj import Transformer
from LCZ4py.local.lcz_krige import krige_predict, rbf_predict, idw_predict

# one representative hour from the heatwave
hour_df = df_event[df_event["date"] == "2019-07-25 14:00:00"].copy()

# sample LCZ class at each station directly from the raster (map CRS is EPSG:4326)
with rasterio.open(lcz_map_path) as src:
    hour_df["lcz"] = [v[0] for v in src.sample(zip(hour_df["Longitude"], hour_df["Latitude"]))]

# project station coordinates to UTM 33N (Berlin)
tx = Transformer.from_crs("EPSG:4326", "EPSG:32633", always_xy=True)
hour_df["x"], hour_df["y"] = tx.transform(hour_df["Longitude"].to_numpy(), hour_df["Latitude"].to_numpy())

# small manual grid over the station bounding box
grid_x = np.linspace(hour_df["x"].min(), hour_df["x"].max(), 40)
grid_y = np.linspace(hour_df["y"].min(), hour_df["y"].max(), 40)

krige_res = krige_predict(
    hour_df, "x", "y", "airT", "lcz", grid_x, grid_y,
    model="Sph", enable_lcz_drift=True, nlags=10,
)
rbf_grid = rbf_predict(hour_df["x"].to_numpy(), hour_df["y"].to_numpy(), hour_df["airT"].to_numpy(), grid_x, grid_y)
idw_grid = idw_predict(hour_df["x"].to_numpy(), hour_df["y"].to_numpy(), hour_df["airT"].to_numpy(), grid_x, grid_y)

print("Kriging (EDK)  range:", round(float(np.nanmin(krige_res.prediction)), 2), "-", round(float(np.nanmax(krige_res.prediction)), 2))
print("RBF            range:", round(float(np.nanmin(rbf_grid)), 2), "-", round(float(np.nanmax(rbf_grid)), 2))
print("IDW            range:", round(float(np.nanmin(idw_grid)), 2), "-", round(float(np.nanmax(idw_grid)), 2))
print("Mean kriging variance:", round(float(np.nanmean(krige_res.variance)), 3))

Kriging (EDK)  range: 31.23 - 34.51
RBF            range: 29.2 - 36.57
IDW            range: 30.9 - 34.28
Mean kriging variance: 0.382


**What to look for:** the EDK prediction range is usually the widest of the
three — because it can push predictions above/below what nearby stations alone would suggest,
whenever the LCZ drift term says "this pixel is Compact Midrise, expect it warmer." RBF and IDW
only ever produce values between the min and max of the *observed* data (both are convex
combinations / smooth interpolants), so they systematically flatten urban-rural contrasts that
fall between station locations. The kriging **variance** is highest away from stations and
near LCZ-class boundaries — exactly where we should trust the map least.

## Step 3 — Gridded interpolation: `lcz_interp_map`

`lcz_interp_map(x, data_frame, var, station_id, sp_res=500.0, method="krige", vg_model="Sph",
LCZinterp=True, n_jobs=-1)` wraps the same primitives above into a production pipeline: it
builds a full north-up UTM grid across the LCZ map's extent, resamples the LCZ raster onto that
grid (for the drift term *and* to mask the output to the map's real footprint), fits/solves the
kriging system per time step in parallel worker processes, and writes a multi-band GeoTIFF (one
band per hour) — returning the output path.

We use a coarse `sp_res=500` (metres) to keep the demo fast, `method="krige"` with
`LCZinterp=True` as the headline External-Drift-Kriging case, then briefly repeat with
`method="idw"` for comparison (no LCZ drift, no variogram fitting — Ordinary Kriging is the
right default here; a small station count and only 4 time steps is exactly its sweet spot).

In [5]:
from LCZ4py import lcz_interp_map

interp_krige_path = lcz_interp_map(
    lcz_map_path, df_event, var="airT", station_id="station",
    sp_res=500.0, method="krige", vg_model="Sph", LCZinterp=True,
    lang="en",
)
print("Kriging (EDK) raster:", interp_krige_path)

interp_idw_path = lcz_interp_map(
    lcz_map_path, df_event, var="airT", station_id="station",
    sp_res=500.0, method="idw",
    lang="en",
)
print("IDW raster (comparison):", interp_idw_path)

06:27:25 - LCZ4py.local.lcz_interp_map - WARNING - LCZ classes available but method={arg idw} ignores them — use method='krige' for External Drift Kriging.


Kriging (EDK) raster: /var/folders/x0/lh_r13_n6_gd2h16vhk2s_3r0000gn/T/tmpf2sb1lt3.tif


IDW raster (comparison): /var/folders/x0/lh_r13_n6_gd2h16vhk2s_3r0000gn/T/tmps383hhot.tif


Both calls return a path to a temporary (or, with `isave=True`, permanently
saved) multi-band GeoTIFF — one band per hourly time step, each named by its timestamp. The
kriging raster is the one we carry forward; the IDW raster is here purely as a sanity-check
comparison of methods.

## Step 4 — Visualizing the interpolated surface: `lcz_plot_interp`

`lcz_plot_interp(x, palette="muted", direction=1)` reads the multi-band raster stack from
`lcz_interp_map` and renders it as a native static matplotlib map of the first band (300 dpi
PNG/PDF/SVG/TIFF written directly, no browser or Kaleido needed). Set `facet=True` for a
small-multiple grid with one panel per hourly time step; pass `renderer="plotly"` for the
interactive Plotly map with the per-band dropdown menu and hover read-out.

In [ ]:
from LCZ4py import lcz_plot_interp

fig = lcz_plot_interp(interp_krige_path, palette="muted", lang="en")
display(fig)

**What to look for:** warmer pockets should visually line up with denser,
higher-LCZ-class-1/2/3 urban fabric (compact/high-rise), and cooler patches with parks, water,
and low-density LCZ classes (11–17) — this LCZ-structured pattern, rather than smooth concentric
rings around any single point, is the visual fingerprint of External Drift Kriging working as
intended.

## Step 5 — Cross-validating the map before trusting it: `lcz_interp_eval`

`lcz_interp_eval(x, data_frame, var, station_id, LOOCV=True, split_ratio=0.8, sp_res=500.0,
vg_model="Sph")` is the essential final check. With `LOOCV=True` (default), for every time step
it leaves each station out in turn, re-krige with the remaining stations, and predicts back at
the held-out station's location — a true leave-one-station-out cross-validation. It returns a
`pl.DataFrame` with `observed`, `predicted`, `residual`, and per-time-step `rmse`/`mae`. Setting
`LOOCV=False` instead does a single random hold-out split (`split_ratio` controls the train
fraction) — faster but noisier for small networks like this one.

## Publication-ready interpolation maps with `style`

The plotting functions now support `style="default"`, `style="nature"`, `style="science"`, and `style="generic_bw"`. The journal presets adjust font family, physical sizing in millimeters, DPI (300 for the publication-oriented presets), and color choices so interpolated surfaces can be exported as manuscript-ready figures.

Because `lcz_plot_interp` is a map-style output, you can also add automatic cartographic helpers with `add_scalebar=True` and `add_north_arrow=True`.

A compact publication-ready example:

- `style="science"` for the interpolated surface.
- `isave=True` with `save_extension="png"` or `"pdf"` to export the final figure.


In [ ]:
from LCZ4py import lcz_plot_interp

fig_pub = lcz_plot_interp(
    interp_krige_path,
    palette="muted",
    direction=1,
    style="science",
    isave=True,
    save_extension="png",
    add_scalebar=True,
    add_north_arrow=True,
    lang="en",
)
display(fig_pub)


In [7]:
from LCZ4py import lcz_interp_eval

eval_df = lcz_interp_eval(
    lcz_map_path, df_event, var="airT", station_id="station",
    LOOCV=True, sp_res=500.0, vg_model="Sph",
    lang="en",
)
eval_df

station,date,observed,predicted,residual,rmse,mae
str,datetime[μs],f64,f64,f64,f64,f64
"""buch""",2019-07-25 12:00:00,32.8,33.2687,-0.4687,1.01315,0.832062
"""dahlem""",2019-07-25 12:00:00,32.2,33.6618,-1.4618,1.01315,0.832062
"""jagen91""",2019-07-25 12:00:00,32.4669,33.858822,-1.391922,1.01315,0.832062
"""marzahn""",2019-07-25 12:00:00,32.9,33.243721,-0.343721,1.01315,0.832062
"""albrecht""",2019-07-25 12:00:00,33.2837,33.661812,-0.378112,1.01315,0.832062
…,…,…,…,…,…,…
"""airportthf""",2019-07-25 15:00:00,33.0,32.448822,0.551178,1.296283,0.97371
"""airporttxl""",2019-07-25 15:00:00,32.6,32.448444,0.151556,1.296283,0.97371
"""rothenburg""",2019-07-25 15:00:00,34.7336,32.182747,2.550853,1.296283,0.97371


In [8]:
summary = eval_df.select(["date", "rmse", "mae"]).unique().sort("date")
summary

date,rmse,mae
datetime[μs],f64,f64
2019-07-25 12:00:00,1.01315,0.832062
2019-07-25 13:00:00,1.091093,0.863892
2019-07-25 14:00:00,0.848124,0.727606
2019-07-25 15:00:00,1.296283,0.97371


**Reading the cross-validation table:** RMSE/MAE on the order of a fraction of
a degree to ~1 °C for a dense urban network like this one indicates the External Drift Kriging
map is trustworthy for downstream heat-exposure or UHI analysis; RMSE approaching the
station-to-station temperature *range* itself would be a red flag that the variogram/drift
model is not capturing the real spatial structure (too few stations, too short a fitted range,
or LCZ classes that don't actually explain much of the variance for this particular event) — in
that case, prefer `lcz_interp_map_plus` (machine-learning interpolation with urban canopy
parameters, notebook 05) over kriging.

## Conclusion

This notebook covered the full classical geostatistical interpolation workflow: quantifying
spatial autocorrelation with `lcz_variogram`, seeing the three low-level scatter-to-grid
primitives (`krige_predict`, `rbf_predict`, `idw_predict`) directly, producing a full gridded
temperature surface with LCZ-aware External Drift Kriging via `lcz_interp_map`, visualizing it
with `lcz_plot_interp`, and — most importantly — validating its accuracy before trusting it
with `lcz_interp_eval`.

This builds directly on **03 · Urban Heat Island** (which compared station-level urban vs.
rural means) by turning those point comparisons into a continuous, defensible surface. Next in
the series: **05 · ML Interpolation & Urban Canopy Parameters** (`05_ml_interpolation_ucp`),
which replaces the geostatistical model with a Random-Forest/OLS model trained on urban
morphological parameters — a better choice when station density is too low for kriging's
distance-based assumptions to hold.